In [ ]:
import sys

sys.path.append("..")
from wavelet_transform.utils.new_bubbledynamics import BubbleDynamics as BD
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import scipy.signal as sp

from wavelet_transform.utils.extensions import fft, to_dB, trim_data
from wavelet_transform.utils.preprocessing import preprocess

In [ ]:
pac = 0.5e6  # the amplitude of the sine wave in Pa
f0_frequency = 1.1* 1e6  # insonation frequency (Hz)
period = 1 / f0_frequency  # the period of the acoustic wave at the fundamental frequency 
fs = f0_frequency * 1e3  # sampling rate to generate the time vector

time_delay = 4 * period  # time delay, t1
pulse_width = 100 * period  # pulse width, W = 1 / Q
phase_initial = 0  #-0.325 * period  # tau
phase_shift = 0  #np.pi / 4  # phi
no_harmonics = 1  # k

R_init = 5e-6  # initial bubble radius in meters
Rdot_init = 0  # initial bubble's wall velocity in m/s

r_measure = 1e-4 # distance from the centre of bubble which radiated pressure (acoustic emission from bubble) is calculated over time

no_cycle = 20  # no. of cycles of the sine wave
deltat = 1.0 / fs
tmax = no_cycle * period
time = np.arange(
    0.0, tmax, deltat
)  # time vector for computing the sine wave, in seconds

In [ ]:
BD_obj = BD(
    R_init=R_init,
    Rdot_init = Rdot_init,
    frequency = f0_frequency,
    time = time,
    deltat = deltat,
)

In [ ]:
BD_obj.initialize()
BD_obj.add_pac_wave(
    f0_frequency, 
    pac, 
    time_delay, 
    pulse_width, 
    phase_initial, 
    phase_shift, 
    no_harmonics
)

In [ ]:
plt.plot(BD_obj.time, BD_obj.npac_wave)
plt.title("Sonication pulse")
plt.xlabel("time/s")
plt.ylabel("amplitude")
plt.show()

In [ ]:
acd_freq = 100e6
acd_pac = 0.1e6
acd_time_delay = 4.0e-6
acd_pulse_width = 10 / acd_freq

for i in range(1):
    acd_time_delay += 1e-6
    BD_obj.add_pac_wave(acd_freq, acd_pac, acd_time_delay, acd_pulse_width, 0, 0, 1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=BD_obj.time, y=BD_obj.get_npac_wave(-1), mode="lines"))
fig.update_layout(
    title="acd wave",
    xaxis_title="time/s",
    yaxis_title="amplitude"
)
fig.show()

plt.plot(BD_obj.time, BD_obj.npac_wave)
plt.title("total wave")
plt.xlabel("time/s")
plt.ylabel("amplitude")
plt.show()

In [ ]:
BD_obj.solver()

In [ ]:
plt.plot(BD_obj.time, BD_obj.R)
plt.title("bubble radius vs time")
plt.xlabel("time/s")
plt.ylabel("amplitude")
plt.show()

In [ ]:
BD_obj.calculate_p_radiated_vokurka(r_measure)
pressure = BD_obj.p_radiated_vokurka
fig = go.Figure()
fig.add_trace(go.Scatter(x=BD_obj.time, y=pressure, mode="lines"))
fig.update_layout(
    title="radiated pressure",
    xaxis_title="time/s",
    yaxis_title="pressure/Pa"
)

In [ ]:
pressure_trimmed, t_trimmed = trim_data(
    pressure, BD_obj.time, [4.6e-6, 5.4e-6], return_time=True
)
pressure_trimmed = preprocess(pressure_trimmed)
pressure_autocor = sp.correlate(
    pressure_trimmed, pressure_trimmed, mode="full", method="auto"
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=np.linspace(0,len(pressure_autocor),len(pressure_autocor)), 
        y=pressure_autocor, 
        mode="lines"
    )
)
fig.update_layout(
    title="autocorrelation of radiated pressure",
    xaxis_title="time/s",
    yaxis_title="pressure/Pa"
)
fig.show()

plt.figure()
plt.plot(t_trimmed, pressure_trimmed)
plt.title("trimmed radiated pressure")
plt.xlabel("time/s")
plt.ylabel("pressure/Pa")
plt.show()

In [ ]:
len(pressure_trimmed)

In [ ]:
fft_results, freq = fft(pressure_trimmed, sampling_rate=fs, fundamental_freq=acd_freq)
fft_results = np.abs(fft_results)

fig = go.Figure()
fig.add_trace(go.Scatter(x=freq, y=to_dB(fft_results), mode="lines", name="fft"))
fig.update_layout(
    title="fft of radiated pressure",
    xaxis=dict(title="Freq [f/f<sub>0</sub>]", range=[0, 5]),
    yaxis=dict(title="Amplitude [dB]", range=[-40, 40]),
)
fig.show()

In [ ]:
from scipy import signal

Font_Size = 24
Tick_Size = 24
Fig_Height = 400
Fig_Width = 800
fft_lim = 200

fsize = 20  # font size
tick_size = 16 # tick size
cl_label_fsize = 14 # clusters labels font size

def stft_plt(data,frequency, period, sampling_rate):
    fig, ax = plt.subplots()
    fig.set_size_inches(18, 9)
    f_stft, t_stft, mag_stft = signal.stft(
        data, sampling_rate, nperseg=2**11, detrend=False, scaling='spectrum'
    )
    stft_plt = ax.pcolormesh(
        t_stft/period, f_stft/frequency, 20*np.log10(np.abs(mag_stft)), 
        vmin=-100, vmax=1, shading='gouraud'
    )
    ax.set_ylabel("$f / f_0$ [DL]", fontsize=fsize)
    ax.set_xlabel("t/T [DL]", fontsize=fsize)  # noqa
    ax.set_title('STFT - spectrum', fontsize=fsize)
    ax.set_ylim([0,fft_lim]) 
    plt.xticks(fontsize=tick_size)
    plt.yticks(np.arange(0,100),np.arange(0,100),fontsize=tick_size)
    cbar_label = ("Magnitude [dB]")
    fig.tight_layout()
    cbar = fig.colorbar(stft_plt, pad=0.01)
    # cbar.ax.tick_params(labelsize=tick_size)
    cbar.set_label(label=cbar_label, size=fsize)
    plt.show()

stft_plt(pressure_trimmed, f0_frequency, period, fs)